# 🎙️ YouTube → Transcripción + SRT

Pegá una o varias URLs de YouTube. El notebook descarga el audio, lo transcribe con **Whisper Large-v3 Turbo** y prepara un ZIP con los archivos `.txt` y `.srt`.

> **Antes de empezar:** en Colab elegí `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU` (o cualquier GPU). Usá únicamente contenido propio o para el que tengas autorización.

In [ ]:
# Instalación (se ejecuta una sola vez por sesión)
!pip -q install -U transformers accelerate gradio yt-dlp sentencepiece
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
import os
import re
import shutil
import time
from pathlib import Path

import gradio as gr
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

WORKDIR = Path('/content/whisper_resultados')
AUDIO_DIR = WORKDIR / 'audio'
OUT_DIR = WORKDIR / 'transcriptos'
MODEL_ID = 'openai/whisper-large-v3-turbo'
asr = None

def cargar_modelo():
    global asr
    if asr is not None:
        return
    if not torch.cuda.is_available():
        raise gr.Error('No se detectó GPU. En Colab activá una GPU y volvé a ejecutar esta celda.')
    dtype = torch.float16
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True
    ).to('cuda')
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    asr = pipeline(
        'automatic-speech-recognition', model=model, tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor, torch_dtype=dtype, device=0,
        chunk_length_s=30
    )

def seguro(nombre):
    nombre = re.sub(r'[^\w .-]+', '_', nombre, flags=re.UNICODE).strip()
    return nombre[:100] or 'video'

def tiempo_srt(segundos):
    ms = round((segundos - int(segundos)) * 1000)
    h, resto = divmod(int(segundos), 3600)
    m, s = divmod(resto, 60)
    return f'{h:02}:{m:02}:{s:02},{ms:03}'

def escribir_srt(chunks, destino):
    lineas = []
    for i, chunk in enumerate(chunks, 1):
        inicio, fin = chunk['timestamp']
        if inicio is None: continue
        if fin is None: fin = inicio + 2
        lineas += [str(i), f'{tiempo_srt(inicio)} --> {tiempo_srt(fin)}', chunk['text'].strip(), '']
    destino.write_text('\n'.join(lineas), encoding='utf-8')

def procesar(urls, idioma, progreso=gr.Progress()):
    lista = [u.strip() for u in urls.splitlines() if u.strip() and not u.strip().startswith('#')]
    if not lista:
        raise gr.Error('Pegá al menos una URL, una por línea.')
    progreso(0, desc='Preparando Whisper Large-v3 Turbo…')
    cargar_modelo()
    if WORKDIR.exists(): shutil.rmtree(WORKDIR)
    AUDIO_DIR.mkdir(parents=True)
    OUT_DIR.mkdir()
    import yt_dlp
    ydl_opts = {
        'format': 'bestaudio/best', 'outtmpl': str(AUDIO_DIR / '%(id)s.%(ext)s'),
        'quiet': True, 'noplaylist': True, 'postprocessors': []
    }
    hechos, errores = [], []
    for n, url in enumerate(lista, 1):
        try:
            progreso((n-1)/len(lista), desc=f'Video {n}/{len(lista)}: descargando audio…')
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                audio = Path(ydl.prepare_filename(info))
            titulo = seguro(info.get('title', f'video_{n}'))
            progreso((n-.45)/len(lista), desc=f'Video {n}/{len(lista)}: transcribiendo…')
            kwargs = {'return_timestamps': True, 'generate_kwargs': {'task': 'transcribe'}}
            if idioma != 'Detectar automáticamente': kwargs['generate_kwargs']['language'] = idioma
            resultado = asr(str(audio), **kwargs)
            txt = OUT_DIR / f'{titulo}.txt'
            srt = OUT_DIR / f'{titulo}.srt'
            txt.write_text(resultado['text'].strip(), encoding='utf-8')
            escribir_srt(resultado.get('chunks', []), srt)
            hechos.append(titulo)
        except Exception as e:
            errores.append(f'• URL {n}: {str(e)[:180]}')
    zip_path = Path('/content/transcriptos_whisper.zip')
    if zip_path.exists(): zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', OUT_DIR)
    progreso(1, desc='¡Terminado!')
    estado = f'### ✅ Listo: {len(hechos)} de {len(lista)} video(s) procesado(s)'
    if hechos: estado += '\n\n**Incluidos:** ' + ', '.join(hechos)
    if errores: estado += '\n\n### ⚠️ Algunos enlaces no se pudieron procesar\n' + '\n'.join(errores)
    return estado, str(zip_path)

CSS = '''
.gradio-container {max-width: 920px !important; background: #f8fafc;}
#hero {text-align:center; padding: 18px 8px 6px;}
#hero h1 {font-size: 2.25rem; margin-bottom: 4px;}
.primary-btn {background: linear-gradient(135deg,#e11d48,#be123c)!important; border:none!important; font-size:1.1rem!important;}
'''

with gr.Blocks(theme=gr.themes.Soft(primary_hue='rose'), css=CSS, title='YouTube a texto') as demo:
    gr.HTML('<div id="hero"><h1>🎙️ YouTube a texto</h1><p>Transcripciones y subtítulos con Whisper Large-v3 Turbo</p></div>')
    with gr.Row():
        with gr.Column(scale=3):
            urls = gr.Textbox(label='URLs de YouTube', lines=9, placeholder='https://www.youtube.com/watch?v=...\nhttps://youtu.be/...', info='Una URL por línea. Podés pegar varias a la vez.')
        with gr.Column(scale=1):
            idioma = gr.Dropdown(['Detectar automáticamente','es','en','pt','fr','it','de'], value='es', label='Idioma', info='Elegí “detectar” si varían.')
            gr.Markdown('#### El ZIP incluye\n- `video.txt` — texto completo\n- `video.srt` — subtítulos con tiempos')
    boton = gr.Button('✨ Procesar transcripciones', variant='primary', elem_classes='primary-btn')
    estado = gr.Markdown()
    descarga = gr.File(label='Descargá tus resultados', file_types=['.zip'], interactive=False)
    boton.click(procesar, inputs=[urls, idioma], outputs=[estado, descarga])

demo.launch(share=False, debug=True)
